# SP-01 Momentum Policy — `01_momentum_policy.ipynb`

First specialised policy of the **Personalized AI Trading Agent** research project. A later Meta-RL agent will choose among five specialised policies; **this model does not select strategies**. It only returns a trading action and a confidence for one state.

| Item | Value |
|---|---|
| Model | SP-01 Momentum Policy (GRU/LSTM sequence encoder + static branch + action head) |
| Action space | `0 = HOLD`, `1 = BUY`, `2 = SELL`, `3 = EXIT` |
| Main artifact | `momentum_policy.pt` (plus `momentum_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json`, `equity_curve.csv`) |
| Execution convention | decision at the **close** of bar *t*, fill at the **open** of bar *t+1* |

> **Research / paper-trading only.** Nothing here is investment advice or a claim of profitability. Any performance number is only meaningful as out-of-sample evidence on the chronological test split, and even that covers one market regime and one set of symbols.

### Important honesty note about the labels
No genuine trader demonstrations are supplied by default. The training labels are therefore **strategy-derived proxy labels** generated by an explicitly documented rule-based momentum strategy (section 5). They are **not human demonstrations**. High classification accuracy against these labels only shows that the network has distilled the rule; it says nothing about profitability. If you later have real trader labels, set `CONFIG['trader_labels_csv']` and the same pipeline performs true behaviour cloning.

### Notebook map
1. Install  2. Configuration  3. Data download  4. Feature engineering (+ look-ahead test)  5. Labels, splits and dataset preparation  6. Model definition  7. Training  8. Calibration and classification metrics  9. Chronological backtest  10. Plots  11. Artifact export  12. Quick-start inference  13. Compatibility summary

## 1. Install
Colab already ships PyTorch, pandas, NumPy, scikit-learn and Matplotlib. Only `yfinance` is upgraded. All indicators are implemented manually (no `pandas-ta`, which is fragile on new NumPy/Python versions).

In [ ]:
import subprocess, sys

def _pip_install(*packages):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])
    except Exception as exc:  # noqa: BLE001 - installation problems must not hide the real error later
        print(f"WARNING: pip install {packages} failed: {exc!r}. Continuing with the installed version, if any.")

_pip_install("yfinance")

## 2. Configuration
Everything configurable lives in `CONFIG`: symbols, benchmark, date range, chronological split dates, window length, costs and hyper-parameters.

* `train_end` / `val_end` define **chronological** splits: train ≤ `train_end` < validation ≤ `val_end` < test.
* `strict_window_isolation=True` additionally requires every *input window* of a validation/test sample to start inside that split, so no window straddles a split boundary.
* `allow_short=True` enables `SELL` (open short) from a flat position. Set `False` for a long-only variant (then `SELL` is never valid).
* `end_date` is exclusive in yfinance.

In [ ]:
import os, json, math, time, random, pickle, warnings, platform, importlib, shutil, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as Fnn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)

CONFIG = {
    "seed": 42,
    # ---- data ------------------------------------------------------------
    "symbols": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "XOM", "JNJ", "PG", "V", "HD"],
    "benchmark": "SPY",
    "start_date": "2010-01-01",
    "end_date": "2025-12-31",          # exclusive
    "min_history_rows": 800,           # symbols with fewer clean daily bars are skipped (and reported)
    "local_csv_dir": None,             # optional fallback: <dir>/<SYMBOL>.csv with Date,Open,High,Low,Close,Volume (real data only)
    "trader_labels_csv": None,         # optional genuine labels: columns symbol,date,action,position[,risk_tolerance,patience,aggressiveness]
    # ---- chronological split (calendar dates) ----------------------------
    "train_end": "2020-12-31",
    "val_end": "2022-12-31",
    # ---- sequence / policy -----------------------------------------------
    "seq_len": 30,
    "strict_window_isolation": True,
    "allow_short": True,
    # ---- backtest --------------------------------------------------------
    "fee_bps": 5.0,                    # per side, on traded notional
    "slippage_bps": 5.0,               # per side, on traded notional
    "short_borrow_bps_annual": 50.0,
    "risk_free_annual": 0.0,
    # ---- model / training -----------------------------------------------
    "model": {"encoder": "gru", "hidden_size": 64, "num_layers": 2, "dropout": 0.25,
              "static_hidden": 16, "head_hidden": 64},
    "train": {"epochs": 40, "batch_size": 512, "lr": 1e-3, "weight_decay": 1e-4, "patience": 7,
              "grad_clip": 1.0, "input_noise_std": 0.02, "max_class_weight": 10.0, "scaler_clip": 5.0},
    "artifact_dir": "/content/sp01_momentum_artifacts" if Path("/content").exists() else "./sp01_momentum_artifacts",
}

assert pd.Timestamp(CONFIG["start_date"]) < pd.Timestamp(CONFIG["train_end"]) < pd.Timestamp(CONFIG["val_end"]) < pd.Timestamp(CONFIG["end_date"]), \
    "dates must satisfy start < train_end < val_end < end"

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART_DIR = Path(CONFIG["artifact_dir"]); ART_DIR.mkdir(parents=True, exist_ok=True)

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else " (CPU fallback)"))
print(f"artifact dir: {ART_DIR.resolve()}")

## 3. Data download
Daily OHLCV from Yahoo Finance via `yfinance` with **adjusted** prices (`auto_adjust=True`, so Open/High/Low/Close are all split/dividend adjusted). Failures are retried, then reported explicitly: a failed symbol is **skipped, never fabricated**. If the benchmark cannot be downloaded or fewer than three symbols succeed, the notebook stops with a clear error.

Cleaning rules (all counted in the report below): duplicate dates removed, non-positive or missing OHLC rows dropped, rows with `High < Low` dropped, negative volume set to missing. Missing volume is kept as missing (never invented) and handled later by explicit train-median imputation.

In [ ]:
import yfinance as yf

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = df.columns.get_level_values(0)
    return df

def download_symbol(sym: str, start: str, end: str, retries: int = 3, local_dir=None):
    """Return (raw_df, source). Raises RuntimeError with the reason if nothing real can be obtained."""
    last_err = "unknown error"
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(sym, start=start, end=end, interval="1d", auto_adjust=True, progress=False, threads=False)
            df = _flatten_columns(df) if df is not None else None
            if df is not None and len(df) > 0:
                return df, "yfinance"
            last_err = "empty result"
        except Exception as exc:  # noqa: BLE001
            last_err = repr(exc)
        time.sleep(1.5 * attempt)
    if local_dir:
        path = Path(local_dir) / f"{sym}.csv"
        if path.exists():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            return df.loc[start:end], f"local_csv:{path}"
    raise RuntimeError(f"{sym}: download failed after {retries} attempts ({last_err})")

def clean_ohlcv(df: pd.DataFrame, sym: str):
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise ValueError(f"{sym}: missing columns {missing}")
    stats = {"symbol": sym, "raw_rows": int(len(df))}
    df = df[OHLCV].apply(pd.to_numeric, errors="coerce")
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df.index = pd.DatetimeIndex(idx).normalize()
    df.index.name = "Date"
    stats["duplicate_dates_removed"] = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")].sort_index()
    price = ["Open", "High", "Low", "Close"]
    bad = df[price].isna().any(axis=1) | (df[price] <= 0).any(axis=1) | (df["High"] < df["Low"])
    stats["bad_price_rows_dropped"] = int(bad.sum())
    df = df.loc[~bad].copy()
    df.loc[df["Volume"] < 0, "Volume"] = np.nan
    stats["missing_volume_rows"] = int(df["Volume"].isna().sum())
    stats["zero_volume_rows"] = int((df["Volume"] == 0).sum())
    stats["extreme_moves_gt_40pct"] = int((df["Close"].pct_change().abs() > 0.40).sum())
    stats["max_gap_days"] = int(df.index.to_series().diff().dt.days.max()) if len(df) > 1 else 0
    stats["rows"] = int(len(df))
    stats["first_date"] = str(df.index.min().date()) if len(df) else None
    stats["last_date"] = str(df.index.max().date()) if len(df) else None
    return df, stats

download_log, raw_data = [], {}

# benchmark first: it is mandatory
try:
    _b, _src = download_symbol(CONFIG["benchmark"], CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
    bench_df, _st = clean_ohlcv(_b, CONFIG["benchmark"])
    _st.update(source=_src, status="ok"); download_log.append(_st)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(f"Benchmark {CONFIG['benchmark']} could not be obtained, cannot continue: {exc}") from exc

for sym in CONFIG["symbols"]:
    try:
        _raw, _src = download_symbol(sym, CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
        _df, _st = clean_ohlcv(_raw, sym)
        if len(_df) < CONFIG["min_history_rows"]:
            raise ValueError(f"only {len(_df)} clean rows (< min_history_rows={CONFIG['min_history_rows']})")
        _st.update(source=_src, status="ok")
        raw_data[sym] = _df
    except Exception as exc:  # noqa: BLE001
        _st = {"symbol": sym, "status": f"SKIPPED: {exc}"}
        print(f"WARNING: {sym} skipped -> {exc}")
    download_log.append(_st)

if len(raw_data) < 3:
    raise RuntimeError(f"Only {len(raw_data)} symbols downloaded successfully; need at least 3. Check the network / symbols.")

report = pd.DataFrame(download_log).set_index("symbol")
print(f"\nSymbols used ({len(raw_data)}): {list(raw_data)}")
print(f"Symbols skipped: {[s for s in CONFIG['symbols'] if s not in raw_data] or 'none'}")
report

## 4. Feature engineering
Feature code lives in a small module, `sp01_features.py`, written to disk by the next cell. Doing it this way means training and inference use **exactly the same code**, and the module ships with the artifacts so the Meta-RL repo can import it. It contains no network or training code.

**Sequence input (30 features per bar, exact order):**
returns (`ret_1, logret_1, ret_5, ret_10, ret_20, ret_60`) · momentum (`rsi_14, macd_pct, macd_signal_pct, macd_hist_pct`) · moving-average relations (`close_ema20, close_ema50, ema20_ema50, close_sma200, sma50_sma200`) · trend strength (`adx_14, di_diff`) · volatility (`atr_pct, rv_20, rv_60`) · volume (`vol_chg_1, vol_ratio_20, vol_z_60`) · candle shape (`hl_range, gap_open, close_loc`) · relative strength / market context (`rel_strength_20, rel_strength_60, bench_ret_20, bench_rv_20`).

Raw OHLCV columns are kept in the dataset (needed for execution prices) but are **not** model inputs: absolute price levels are non-stationary, so the model sees scale-free derived features instead.

**Static input (6 dims):** position one-hot `[flat, long, short]` + trader profile `[risk_tolerance, patience, aggressiveness]` in `[0, 1]`.

Every feature at row *t* uses only data up to the close of bar *t*. A perturbation test below proves it: it scrambles all future prices and verifies that earlier features do not change.

In [ ]:
%%writefile sp01_features.py
"""SP-01 Momentum Policy - shared constants and causal feature engineering.

This module never touches the network and never trains anything. Every feature at
row t is computed from data up to and including the close of bar t.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Common action space (shared by all five specialised policies)
# ----------------------------------------------------------------------------
HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTION_NAMES = ["HOLD", "BUY", "SELL", "EXIT"]
N_ACTIONS = 4

# Position state: +1 long, 0 flat, -1 short. One-hot order below is part of the schema.
POSITION_ORDER = (0, 1, -1)                      # one-hot index -> position value
POSITION_NAMES = ["pos_flat", "pos_long", "pos_short"]

# Trader-profile features, each in [0, 1].
PROFILE_FEATURES = ["risk_tolerance", "patience", "aggressiveness"]
DEFAULT_PROFILE = (0.5, 0.5, 0.5)

STATIC_FEATURE_NAMES = POSITION_NAMES + PROFILE_FEATURES   # 3 + 3 = 6 dims

# Sequential market features: exact column order of the model input (30 dims).
FEATURE_COLUMNS = [
    # returns
    "ret_1", "logret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    # momentum oscillators
    "rsi_14", "macd_pct", "macd_signal_pct", "macd_hist_pct",
    # moving-average relationships
    "close_ema20", "close_ema50", "ema20_ema50", "close_sma200", "sma50_sma200",
    # trend strength
    "adx_14", "di_diff",
    # volatility
    "atr_pct", "rv_20", "rv_60",
    # volume
    "vol_chg_1", "vol_ratio_20", "vol_z_60",
    # candle shape
    "hl_range", "gap_open", "close_loc",
    # relative strength vs benchmark and market context
    "rel_strength_20", "rel_strength_60", "bench_ret_20", "bench_rv_20",
]
N_FEATURES = len(FEATURE_COLUMNS)
N_STATIC = len(STATIC_FEATURE_NAMES)
WARMUP_BARS = 200          # sma_200 is the longest look-back
OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]


# ----------------------------------------------------------------------------
# Action validity and position transitions
# ----------------------------------------------------------------------------
def valid_action_mask(position, allow_short: bool = True) -> np.ndarray:
    """Boolean (B, 4) mask of valid actions given the current position.

    flat  -> HOLD, BUY (open long), SELL (open short, only if allow_short)
    long  -> HOLD, EXIT
    short -> HOLD, EXIT
    """
    pos = np.asarray(position).astype(int).reshape(-1)
    mask = np.zeros((pos.size, N_ACTIONS), dtype=bool)
    mask[:, HOLD] = True
    flat = pos == 0
    mask[flat, BUY] = True
    mask[flat, SELL] = bool(allow_short)
    mask[~flat, EXIT] = True
    return mask


def apply_action(position, action) -> np.ndarray:
    """Return the new position after executing (already validated) actions."""
    pos = np.asarray(position).astype(int).reshape(-1)
    act = np.asarray(action).astype(int).reshape(-1)
    new = pos.copy()
    new[(act == BUY) & (pos == 0)] = 1
    new[(act == SELL) & (pos == 0)] = -1
    new[(act == EXIT) & (pos != 0)] = 0
    return new


def profile_array(profile, batch_size: int) -> np.ndarray:
    """Normalise a profile (dict, sequence, (B,3) array or None) to float32 (B, 3)."""
    if profile is None:
        profile = DEFAULT_PROFILE
    if isinstance(profile, dict):
        profile = [profile.get(k, d) for k, d in zip(PROFILE_FEATURES, DEFAULT_PROFILE)]
    arr = np.asarray(profile, dtype=np.float32)
    if arr.ndim == 1:
        arr = np.tile(arr[None, :], (batch_size, 1))
    if arr.shape != (batch_size, len(PROFILE_FEATURES)):
        raise ValueError(f"profile must have shape (3,) or ({batch_size}, 3); got {arr.shape}")
    if np.any(arr < 0) or np.any(arr > 1) or not np.all(np.isfinite(arr)):
        raise ValueError("profile values must be finite and within [0, 1]")
    return arr


def encode_static(position, profile) -> np.ndarray:
    """Static input vector (B, 6) = [flat, long, short one-hot] + [risk, patience, aggressiveness]."""
    pos = np.asarray(position).astype(int).reshape(-1)
    if not np.all(np.isin(pos, POSITION_ORDER)):
        raise ValueError("position must be in {-1, 0, +1}")
    onehot = np.stack([(pos == p) for p in POSITION_ORDER], axis=1).astype(np.float32)
    return np.concatenate([onehot, profile_array(profile, pos.size)], axis=1).astype(np.float32)


# ----------------------------------------------------------------------------
# Indicators (manual implementations, causal)
# ----------------------------------------------------------------------------
def _pct(s: pd.Series, n: int) -> pd.Series:
    return s / s.shift(n) - 1.0


def _wilder(s: pd.Series, n: int) -> pd.Series:
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def rsi(close: pd.Series, n: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    rs = _wilder(gain, n) / _wilder(loss, n)
    return 100.0 - 100.0 / (1.0 + rs)


def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev = close.shift(1)
    return pd.concat([high - low, (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)


def adx_di(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14):
    """Wilder ADX with +DI / -DI."""
    up = high.diff()
    down = -low.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=high.index)
    atr = _wilder(true_range(high, low, close), n)
    plus_di = 100.0 * _wilder(plus_dm, n) / atr
    minus_di = 100.0 * _wilder(minus_dm, n) / atr
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return _wilder(dx, n), plus_di, minus_di


# ----------------------------------------------------------------------------
# Feature builder
# ----------------------------------------------------------------------------
def build_features(ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> pd.DataFrame:
    """Compute causal features for one symbol.

    Parameters
    ----------
    ohlcv : DataFrame indexed by date, columns Open/High/Low/Close/Volume (adjusted prices).
    benchmark : DataFrame indexed by date with at least a Close column.

    Returns a DataFrame with the same index: the OHLCV columns followed by FEATURE_COLUMNS.
    Row t only depends on rows <= t (verified by a perturbation test in the notebook).
    """
    if benchmark is None or "Close" not in benchmark:
        raise ValueError("a benchmark DataFrame with a Close column is required")
    missing = [c for c in OHLCV_COLUMNS if c not in ohlcv.columns]
    if missing:
        raise ValueError(f"ohlcv is missing columns: {missing}")

    df = ohlcv[OHLCV_COLUMNS].astype(float).copy()
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    f = pd.DataFrame(index=df.index)

    # returns
    f["ret_1"] = _pct(c, 1)
    f["logret_1"] = np.log(c).diff()
    for n in (5, 10, 20, 60):
        f[f"ret_{n}"] = _pct(c, n)

    # momentum oscillators
    f["rsi_14"] = rsi(c, 14)
    ema12 = c.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = c.ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    f["macd_pct"] = macd / c
    f["macd_signal_pct"] = macd_signal / c
    f["macd_hist_pct"] = (macd - macd_signal) / c

    # moving-average relationships
    ema20 = c.ewm(span=20, adjust=False, min_periods=20).mean()
    ema50 = c.ewm(span=50, adjust=False, min_periods=50).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    f["close_ema20"] = c / ema20 - 1.0
    f["close_ema50"] = c / ema50 - 1.0
    f["ema20_ema50"] = ema20 / ema50 - 1.0
    f["close_sma200"] = c / sma200 - 1.0
    f["sma50_sma200"] = sma50 / sma200 - 1.0

    # trend strength
    adx, plus_di, minus_di = adx_di(h, l, c, 14)
    f["adx_14"] = adx
    f["di_diff"] = plus_di - minus_di

    # volatility
    f["atr_pct"] = _wilder(true_range(h, l, c), 14) / c
    f["rv_20"] = f["logret_1"].rolling(20).std() * np.sqrt(252.0)
    f["rv_60"] = f["logret_1"].rolling(60).std() * np.sqrt(252.0)

    # volume (zero volume treated as missing for ratios; never fabricated)
    vz = v.where(v > 0)
    f["vol_chg_1"] = np.log(vz / vz.shift(1)).clip(-3.0, 3.0)
    f["vol_ratio_20"] = vz / vz.rolling(20).mean()
    f["vol_z_60"] = (vz - vz.rolling(60).mean()) / vz.rolling(60).std().replace(0.0, np.nan)

    # candle shape
    f["hl_range"] = (h - l) / c
    f["gap_open"] = o / c.shift(1) - 1.0
    f["close_loc"] = (c - l) / (h - l).replace(0.0, np.nan)

    # relative strength vs benchmark (past-only forward fill, at most 3 bars)
    bc = benchmark["Close"].astype(float).reindex(df.index).ffill(limit=3)
    b_ret20 = _pct(bc, 20)
    b_ret60 = _pct(bc, 60)
    f["rel_strength_20"] = f["ret_20"] - b_ret20
    f["rel_strength_60"] = f["ret_60"] - b_ret60
    f["bench_ret_20"] = b_ret20
    f["bench_rv_20"] = np.log(bc).diff().rolling(20).std() * np.sqrt(252.0)

    f = f[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    return pd.concat([df, f], axis=1)


def scale_features(x, median, mean, std, clip) -> np.ndarray:
    """Single implementation of the preprocessing used in training AND inference.

    Non-finite values -> training-set median, then z-score, then clip to +/- clip.
    """
    x = np.asarray(x, dtype=np.float32)
    x = np.where(np.isfinite(x), x, np.asarray(median, dtype=np.float32))
    x = (x - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return np.clip(x, -float(clip), float(clip)).astype(np.float32)


def build_feature_schema(seq_len: int) -> dict:
    """Machine-readable description of the exact model input layout."""
    return {
        "schema_version": "1.0.0",
        "policy_id": "SP-01",
        "sequence_input": {
            "name": "sequence",
            "shape": ["batch", seq_len, N_FEATURES],
            "dtype": "float32",
            "order": "oldest bar first, newest bar last (index -1 is the decision bar)",
            "features": [{"index": i, "name": n} for i, n in enumerate(FEATURE_COLUMNS)],
            "scaling": "NaN -> train median, then (x - mean) / std, then clip to +/- clip (see preprocessing.pkl)",
            "note": "Callers pass RAW (unscaled) features; the policy scales internally.",
        },
        "static_input": {
            "name": "static",
            "shape": ["batch", N_STATIC],
            "dtype": "float32",
            "features": [{"index": i, "name": n} for i, n in enumerate(STATIC_FEATURE_NAMES)],
            "position_encoding": {"one_hot_order": list(POSITION_NAMES), "position_values": list(POSITION_ORDER)},
            "profile_ranges": {k: [0.0, 1.0] for k in PROFILE_FEATURES},
            "default_profile": dict(zip(PROFILE_FEATURES, DEFAULT_PROFILE)),
        },
        "output": {
            "action_probs": ["batch", N_ACTIONS],
            "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
            "confidence": "temperature-calibrated probability of the chosen action after masking invalid actions",
        },
        "valid_actions": {
            "flat": ["HOLD", "BUY", "SELL (only if allow_short)"],
            "long": ["HOLD", "EXIT"],
            "short": ["HOLD", "EXIT"],
        },
        "execution_convention": "decision at close of bar t; fill at open of bar t+1",
    }


In [ ]:
import sp01_features
importlib.reload(sp01_features)
from sp01_features import (ACTION_NAMES, HOLD, BUY, SELL, EXIT, N_ACTIONS, FEATURE_COLUMNS, N_FEATURES, N_STATIC,
                           STATIC_FEATURE_NAMES, PROFILE_FEATURES, DEFAULT_PROFILE, OHLCV_COLUMNS,
                           build_features, build_feature_schema, encode_static, valid_action_mask, apply_action,
                           scale_features)

print(f"{N_FEATURES} sequence features, {N_STATIC} static features")
for i, name in enumerate(FEATURE_COLUMNS):
    print(f"  seq[{i:2d}] {name}")
for i, name in enumerate(STATIC_FEATURE_NAMES):
    print(f"  static[{i}] {name}")

In [ ]:
EVAL_ONLY_COLUMNS = ["fwd_open_ret_5"]   # uses FUTURE prices; for evaluation tables only, never a model input
assert set(EVAL_ONLY_COLUMNS).isdisjoint(FEATURE_COLUMNS)

# ---- causality (no look-ahead) test --------------------------------------------------------
def causality_test(sym: str, frac: float = 0.7) -> None:
    ohlcv = raw_data[sym]; n = int(len(ohlcv) * frac)
    base = build_features(ohlcv, bench_df)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    rng = np.random.default_rng(0)
    pert = ohlcv.copy()
    pert.iloc[n:, :] = pert.iloc[n:, :].to_numpy() * rng.uniform(0.3, 3.0, size=pert.iloc[n:].shape)   # scramble the FUTURE
    bench_p = bench_df.copy()
    future = bench_p.index > ohlcv.index[n - 1]
    bench_p.loc[future, :] = bench_p.loc[future].to_numpy() * 2.0
    alt = build_features(pert, bench_p)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    assert np.allclose(base, alt, equal_nan=True, rtol=1e-9, atol=1e-12), f"{sym}: features depend on future data!"

for _s in list(raw_data)[:2]:
    causality_test(_s)
print("Causality test passed: features at row t are unchanged when all data after t is scrambled.")

# ---- build the chronological panel ---------------------------------------------------------
frames, feature_report = [], []
for sym, ohlcv in raw_data.items():
    f = build_features(ohlcv, bench_df)
    firsts = [f[c].first_valid_index() for c in FEATURE_COLUMNS]
    if any(x is None for x in firsts):
        print(f"WARNING: {sym} skipped: a feature is entirely missing")
        continue
    f = f.loc[max(firsts):].copy()                                  # drop warm-up rows only
    nan_before = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f[FEATURE_COLUMNS] = f[FEATURE_COLUMNS].ffill(limit=3)          # past-only fill, at most 3 bars
    nan_after = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f["symbol"] = sym
    f["fwd_open_ret_5"] = f["Open"].shift(-6) / f["Open"].shift(-1) - 1.0      # EVAL ONLY (future)
    f = f.reset_index().rename(columns={"Date": "date", "index": "date"})
    feature_report.append({"symbol": sym, "rows_after_warmup": len(f), "first_date": f["date"].min().date(),
                           "nan_cells_before_ffill": nan_before, "nan_cells_after_ffill": nan_after})
    frames.append(f)

panel = pd.concat(frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)
panel["row_in_sym"] = panel.groupby("symbol").cumcount()
print(f"\npanel: {panel.shape[0]:,} rows x {panel.shape[1]} columns, {panel['symbol'].nunique()} symbols, "
      f"{panel['date'].min().date()} -> {panel['date'].max().date()}")
print("Residual NaNs in features are imputed with the TRAIN median at scaling time (counted below).")
pd.DataFrame(feature_report).set_index("symbol")

## 5. Labels and dataset preparation

### 5.1 Momentum rule (source of the proxy labels)
For a state *(bar t, current position, trader profile)* the rule uses only information available at the close of bar *t*:

* **Long entry** (flat → BUY) requires **all** of: uptrend (`close > EMA50` and `EMA20 > EMA50`), trend strength (`ADX14 ≥ adx_thr` and `+DI > −DI`), positive past returns (`ret_20 > 0` and `ret_60 > mom_thr`), positive relative strength vs the benchmark (`rel_strength_60 > 0`), volume confirmation (`volume / 20-bar mean ≥ vol_thr`) and volatility not above the profile's cap (`ATR% ≤ atr_cap`).
* **Short entry** (flat → SELL, if `allow_short`) is the mirror image.
* **Exit** (long → EXIT) when the trend breaks (`EMA20 < EMA50`) or the 20-bar return falls below `-exit_tol`; shorts mirror this.
* Otherwise **HOLD**. Invalid actions are never produced (position-dependent masking).

The trader profile modulates the thresholds, so the profile input is meaningful rather than decorative:
`adx_thr = 25 − 10·aggressiveness`, `mom_thr = 0.06 − 0.05·aggressiveness`, `vol_thr = 1 − 0.2·aggressiveness`, `atr_cap = 0.02 + 0.04·risk_tolerance`, `exit_tol = 0.01 + 0.04·patience`.

### 5.2 Sample construction
Because the rule is a function of *(features, position, profile)*, every bar yields **three** training samples (flat / long / short) with a random profile each. This gives the policy correct labels in every position state it can reach during its own backtest (no distribution-shift trap from only seeing the teacher's own position path). Labels depend only on past data, so no label looks into the future.

### 5.3 Splits and preprocessing
* Splits are by calendar date (train ≤ `train_end` < val ≤ `val_end` < test). Samples are only shuffled **inside** the training split, never across splits.
* The scaler (train median → z-score → clip) is fit on **training-period rows only**.
* Class imbalance is handled with inverse-frequency class weights (capped), overfitting with dropout, weight decay, input noise, gradient clipping and early stopping on validation loss.

In [ ]:
D_TRAIN_END = pd.Timestamp(CONFIG["train_end"]); D_VAL_END = pd.Timestamp(CONFIG["val_end"])
SPLIT_BOUNDS = {"train": (None, D_TRAIN_END),
                "val": (D_TRAIN_END + pd.Timedelta(days=1), D_VAL_END),
                "test": (D_VAL_END + pd.Timedelta(days=1), None)}
L = CONFIG["seq_len"]
RAW = {c: panel[c].to_numpy(dtype=float) for c in FEATURE_COLUMNS}               # unscaled, with NaNs
RAW_MAT = panel[FEATURE_COLUMNS].to_numpy(dtype=np.float32)                      # (N, F)
_WIN_START = panel.groupby("symbol")["date"].shift(L - 1)                         # date of the first bar of each window

def eligible_rows(split: str) -> np.ndarray:
    """Global row indices whose decision bar lies in the split and whose L-bar window is complete (and, if
    strict_window_isolation, starts inside the split)."""
    start, end = SPLIT_BOUNDS[split]
    ok = _WIN_START.notna().to_numpy().copy()
    if start is not None:
        ok &= (panel["date"] >= start).to_numpy()
        if CONFIG["strict_window_isolation"]:
            ok &= (_WIN_START >= start).to_numpy()
    if end is not None:
        ok &= (panel["date"] <= end).to_numpy()
    return np.where(ok)[0]

def momentum_rule_labels(feat: dict, pos: np.ndarray, profile: np.ndarray, allow_short: bool = True) -> np.ndarray:
    """Documented momentum rule -> action label (0..3) for each (features, position, profile) triple."""
    risk, pat, agg = profile[:, 0], profile[:, 1], profile[:, 2]
    adx_thr, mom_thr = 25.0 - 10.0 * agg, 0.06 - 0.05 * agg
    vol_thr, atr_cap, exit_tol = 1.0 - 0.2 * agg, 0.02 + 0.04 * risk, 0.01 + 0.04 * pat
    with np.errstate(invalid="ignore"):       # NaN comparisons are False -> HOLD
        up = (feat["close_ema50"] > 0) & (feat["ema20_ema50"] > 0)
        dn = (feat["close_ema50"] < 0) & (feat["ema20_ema50"] < 0)
        strong = feat["adx_14"] >= adx_thr
        vol_ok = feat["vol_ratio_20"] >= vol_thr
        calm = feat["atr_pct"] <= atr_cap
        long_entry = (up & strong & (feat["di_diff"] > 0) & (feat["ret_20"] > 0) & (feat["ret_60"] > mom_thr)
                      & (feat["rel_strength_60"] > 0) & vol_ok & calm)
        short_entry = (dn & strong & (feat["di_diff"] < 0) & (feat["ret_20"] < 0) & (feat["ret_60"] < -mom_thr)
                       & (feat["rel_strength_60"] < 0) & vol_ok & calm)
        long_exit = (feat["ema20_ema50"] < 0) | (feat["ret_20"] < -exit_tol)
        short_exit = (feat["ema20_ema50"] > 0) | (feat["ret_20"] > exit_tol)
    y = np.zeros(len(pos), dtype=np.int64)
    flat = pos == 0
    y[flat & long_entry] = BUY
    if allow_short:
        y[flat & ~long_entry & short_entry] = SELL
    y[(pos == 1) & long_exit] = EXIT
    y[(pos == -1) & short_exit] = EXIT
    return y

def _finalize(rows, pos, prof, y):
    valid = valid_action_mask(pos, CONFIG["allow_short"])
    assert valid[np.arange(len(y)), y].all(), "label contains an action that is invalid for its position"
    return {"row": np.asarray(rows, dtype=np.int64), "pos": np.asarray(pos, dtype=np.int64),
            "profile": np.asarray(prof, dtype=np.float32), "y": np.asarray(y, dtype=np.int64),
            "valid": valid, "static": encode_static(pos, prof)}

def build_rule_samples(split: str, rng: np.random.Generator) -> dict:
    rows = eligible_rows(split)
    r = np.repeat(rows, 3)
    pos = np.tile(np.array([0, 1, -1]), len(rows))
    prof = rng.uniform(0.0, 1.0, size=(len(r), 3)).astype(np.float32)
    y = momentum_rule_labels({k: v[r] for k, v in RAW.items()}, pos, prof, CONFIG["allow_short"])
    return _finalize(r, pos, prof, y)

def build_trader_samples(split: str, csv_path: str) -> dict:
    """Behaviour cloning from genuine trader actions (optional)."""
    tl = pd.read_csv(csv_path)
    need = {"symbol", "date", "action", "position"}
    if not need <= set(tl.columns):
        raise ValueError(f"trader label file needs columns {sorted(need)}; has {list(tl.columns)}")
    tl["date"] = pd.to_datetime(tl["date"]).dt.normalize()
    names = {n: i for i, n in enumerate(ACTION_NAMES)}
    def _act(a):
        s = str(a).strip().upper()
        return names[s] if s in names else int(float(s))
    tl["action"] = tl["action"].map(_act)
    key = panel[["symbol", "date"]].assign(row=np.arange(len(panel)))
    m = tl.merge(key, on=["symbol", "date"], how="inner")
    m = m[m["row"].isin(eligible_rows(split))]
    pos = m["position"].to_numpy(dtype=int)
    if not np.isin(pos, [-1, 0, 1]).all() or not np.isin(m["action"], range(N_ACTIONS)).all():
        raise ValueError("trader labels contain invalid positions or actions")
    ok = valid_action_mask(pos, CONFIG["allow_short"])[np.arange(len(m)), m["action"].to_numpy(dtype=int)]
    if (~ok).any():
        print(f"  [{split}] dropping {(~ok).sum()} trader rows whose action is invalid for the given position")
    m, pos = m[ok], pos[ok]
    if set(PROFILE_FEATURES) <= set(m.columns):
        prof = m[PROFILE_FEATURES].to_numpy(dtype=np.float32)
    else:
        prof = np.tile(np.array(DEFAULT_PROFILE, dtype=np.float32), (len(m), 1))
    if len(m) == 0:
        raise ValueError(f"no usable trader labels in the {split} split")
    return _finalize(m["row"].to_numpy(), pos, prof, m["action"].to_numpy(dtype=int))

rng = np.random.default_rng(CONFIG["seed"])
if CONFIG["trader_labels_csv"]:
    LABEL_SOURCE = "trader_demonstrations"
    SAMPLES = {s: build_trader_samples(s, CONFIG["trader_labels_csv"]) for s in ("train", "val", "test")}
else:
    LABEL_SOURCE = "strategy_derived_proxy"
    SAMPLES = {s: build_rule_samples(s, rng) for s in ("train", "val", "test")}

print("=" * 88)
if LABEL_SOURCE == "strategy_derived_proxy":
    print("LABEL SOURCE: STRATEGY-DERIVED PROXY LABELS from the documented momentum rule. NOT human demonstrations.")
else:
    print("LABEL SOURCE: genuine trader demonstrations (behaviour cloning).")
print("=" * 88)
dist = pd.DataFrame({s: np.bincount(S["y"], minlength=N_ACTIONS) for s, S in SAMPLES.items()}, index=ACTION_NAMES)
print(dist.assign(**{f"{c}_%": (100 * dist[c] / dist[c].sum()).round(1) for c in dist.columns}))
for s, S in SAMPLES.items():
    r = panel.loc[S["row"], "date"]
    print(f"{s:5s}: {len(S['y']):>8,} samples | decision dates {r.min().date()} -> {r.max().date()}")
assert panel.loc[SAMPLES["train"]["row"], "date"].max() < panel.loc[SAMPLES["val"]["row"], "date"].min()
assert panel.loc[SAMPLES["val"]["row"], "date"].max() < panel.loc[SAMPLES["test"]["row"], "date"].min()

# ---- class weights (training split only) --------------------------------------------------
_cnt = np.bincount(SAMPLES["train"]["y"], minlength=N_ACTIONS).astype(float)
_w = np.where(_cnt > 0, _cnt.sum() / (N_ACTIONS * np.maximum(_cnt, 1)), 0.0)
_w = np.minimum(_w, CONFIG["train"]["max_class_weight"])
CLASS_WEIGHTS = (_w * (_cnt.sum() / (_w * _cnt).sum())).astype(np.float32)       # mean sample weight = 1
print("class weights:", {n: round(float(w), 3) for n, w in zip(ACTION_NAMES, CLASS_WEIGHTS)})
if (_cnt == 0).any():
    print("NOTE: classes with no training examples:", [ACTION_NAMES[i] for i in np.where(_cnt == 0)[0]])

# ---- preprocessing fit on TRAIN-PERIOD rows only ------------------------------------------
_fit = (panel["date"] <= D_TRAIN_END).to_numpy()
_med = np.nanmedian(RAW_MAT[_fit], axis=0)
_imp = np.where(np.isfinite(RAW_MAT[_fit]), RAW_MAT[_fit], _med)
_std = _imp.std(axis=0); _std = np.where(_std < 1e-8, 1.0, _std)
PREPROCESSING = {"feature_columns": list(FEATURE_COLUMNS), "median": _med.astype(np.float32),
                 "mean": _imp.mean(axis=0).astype(np.float32), "std": _std.astype(np.float32),
                 "clip": float(CONFIG["train"]["scaler_clip"]),
                 "fit_rows_end_date": str(D_TRAIN_END.date()), "n_fit_rows": int(_fit.sum())}
X_SCALED = scale_features(RAW_MAT, PREPROCESSING["median"], PREPROCESSING["mean"], PREPROCESSING["std"], PREPROCESSING["clip"])
print(f"scaler fit on {PREPROCESSING['n_fit_rows']:,} rows with dates <= {PREPROCESSING['fit_rows_end_date']}; "
      f"non-finite feature cells imputed overall: {int((~np.isfinite(RAW_MAT)).sum()):,}")

## 6. Model definition
The network lives in `sp01_model.py` (written to disk by the next cell, shipped with the artifacts). It has **no data-download or training code**.

* `MomentumPolicyNet`: GRU or LSTM encoder over the last `seq_len` bars → last hidden state, concatenated with a small dense branch for the static input (position one-hot + trader profile) → dense head → 4 action logits.
* `MomentumPolicy`: inference wrapper with `predict(state)`, `save(path)` and `load(path, device)`. It scales raw features with the saved preprocessing, masks invalid actions by position, and returns calibrated probabilities.
* **Confidence** = temperature-calibrated softmax probability of the chosen action after masking invalid actions. The temperature is fit on the validation split against the proxy labels, so confidence means "how strongly the policy agrees with the momentum rule", **not** a probability that the trade will be profitable.

**Input contract of `predict(state)`** (raw, unscaled features):
`state = {"sequence": (L, 30) or (B, L, 30) array in FEATURE_COLUMNS order, oldest bar first, "position": -1 | 0 | +1, "profile": [risk_tolerance, patience, aggressiveness]}`

In [ ]:
%%writefile sp01_model.py
"""SP-01 Momentum Policy - network and inference wrapper.

Contains NO data download and NO training code. Load a saved policy with
``MomentumPolicy.load(dir_or_pt_file, device)`` and call ``predict(state)``.
"""
from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sp01_features import (
    ACTION_NAMES, DEFAULT_PROFILE, FEATURE_COLUMNS, N_ACTIONS, N_FEATURES, N_STATIC,
    PROFILE_FEATURES, WARMUP_BARS, build_feature_schema, build_features, encode_static,
    scale_features, valid_action_mask,
)

ARTIFACT_NAME = "momentum_policy.pt"
MASK_VALUE = -1e9


class MomentumPolicyNet(nn.Module):
    """GRU/LSTM encoder over market features + static (position, profile) branch + action head."""

    def __init__(self, n_features: int = N_FEATURES, n_static: int = N_STATIC, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.25, static_hidden: int = 16,
                 head_hidden: int = 64, encoder: str = "gru", n_actions: int = N_ACTIONS):
        super().__init__()
        if encoder not in ("gru", "lstm"):
            raise ValueError("encoder must be 'gru' or 'lstm'")
        rnn_cls = nn.GRU if encoder == "gru" else nn.LSTM
        self.rnn = rnn_cls(n_features, hidden_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.static = nn.Sequential(nn.Linear(n_static, static_hidden), nn.GELU())
        self.head = nn.Sequential(
            nn.Linear(hidden_size + static_hidden, head_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, n_actions),
        )

    def forward(self, seq: torch.Tensor, static: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn(seq)
        z = torch.cat([out[:, -1, :], self.static(static)], dim=1)
        return self.head(z)


def mask_logits(logits: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill(~valid, MASK_VALUE)


def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


class MomentumPolicy:
    """Inference wrapper: scaling + network + action masking + calibrated confidence."""

    def __init__(self, net: MomentumPolicyNet, config: dict, preprocessing: dict, device="cpu"):
        self.device = torch.device(device)
        self.net = net.to(self.device).eval()
        self.config = _jsonable(config)
        self.pre = {
            "feature_columns": list(preprocessing["feature_columns"]),
            "median": np.asarray(preprocessing["median"], dtype=np.float32),
            "mean": np.asarray(preprocessing["mean"], dtype=np.float32),
            "std": np.asarray(preprocessing["std"], dtype=np.float32),
            "clip": float(preprocessing["clip"]),
        }
        if self.pre["feature_columns"] != list(FEATURE_COLUMNS):
            raise ValueError("feature order in preprocessing does not match sp01_features.FEATURE_COLUMNS")
        self.seq_len = int(self.config["seq_len"])
        self.allow_short = bool(self.config.get("allow_short", True))
        self.temperature = float(self.config.get("temperature", 1.0))

    # ------------------------------------------------------------------ scaling
    def scale(self, seq: np.ndarray) -> np.ndarray:
        """Raw (..., F) -> scaled float32. NaN -> train median, standardise, clip."""
        return scale_features(seq, self.pre["median"], self.pre["mean"], self.pre["std"], self.pre["clip"])

    # ------------------------------------------------------------------ inference
    @torch.no_grad()
    def predict_scaled(self, seq_scaled: torch.Tensor, static: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        """Calibrated probabilities (B, 4) from already-scaled tensors on self.device."""
        logits = mask_logits(self.net(seq_scaled, static), valid)
        return torch.softmax(logits / self.temperature, dim=1)

    @torch.no_grad()
    def predict(self, state: dict) -> dict:
        """Predict an action for one state or a batch of states.

        state keys
        ----------
        sequence : array (L, 30) or (B, L, 30) of RAW features in FEATURE_COLUMNS order,
                   oldest bar first. L >= seq_len; the last seq_len bars are used.
        position : int in {-1, 0, 1} (or length-B array). Default 0 (flat).
        profile  : dict or length-3 / (B, 3) array [risk_tolerance, patience, aggressiveness]
                   in [0, 1]. Default (0.5, 0.5, 0.5).

        Returns a dict with action_probs, action, action_name, confidence, valid_actions, entropy.
        Single-state input yields scalars / 1-D arrays; batch input yields arrays.
        """
        seq = np.asarray(state["sequence"], dtype=np.float32)
        single = seq.ndim == 2
        if single:
            seq = seq[None]
        if seq.ndim != 3 or seq.shape[2] != N_FEATURES:
            raise ValueError(f"sequence must have shape (L, {N_FEATURES}) or (B, L, {N_FEATURES}); got {seq.shape}")
        if seq.shape[1] < self.seq_len:
            raise ValueError(f"need at least seq_len={self.seq_len} bars; got {seq.shape[1]}")
        seq = seq[:, -self.seq_len:, :]
        batch = seq.shape[0]

        position = np.asarray(state.get("position", 0)).astype(int).reshape(-1)
        if position.size == 1 and batch > 1:
            position = np.repeat(position, batch)
        if position.size != batch:
            raise ValueError("position must be scalar or have one entry per sequence")
        static = encode_static(position, state.get("profile", DEFAULT_PROFILE))
        valid = valid_action_mask(position, self.allow_short)

        seq_t = torch.from_numpy(self.scale(seq)).to(self.device)
        static_t = torch.from_numpy(static).to(self.device)
        valid_t = torch.from_numpy(valid).to(self.device)
        probs = self.predict_scaled(seq_t, static_t, valid_t).cpu().numpy()

        action = probs.argmax(axis=1)
        confidence = probs[np.arange(batch), action]
        entropy = -(probs * np.log(np.clip(probs, 1e-12, 1.0))).sum(axis=1)
        names = [ACTION_NAMES[a] for a in action]
        if single:
            return {"action_probs": probs[0], "action": int(action[0]), "action_name": names[0],
                    "confidence": float(confidence[0]), "valid_actions": valid[0], "entropy": float(entropy[0])}
        return {"action_probs": probs, "action": action, "action_name": names,
                "confidence": confidence, "valid_actions": valid, "entropy": entropy}

    def state_from_ohlcv(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame, position: int = 0, profile=None) -> dict:
        """Build a predict() state from OHLCV frames the caller already holds (no downloading)."""
        feats = build_features(ohlcv, benchmark)[FEATURE_COLUMNS]
        if len(feats) < WARMUP_BARS + self.seq_len:
            raise ValueError(f"need at least {WARMUP_BARS + self.seq_len} bars of history; got {len(feats)}")
        window = feats.iloc[-self.seq_len:]
        return {"sequence": window.to_numpy(dtype=np.float32), "position": int(position),
                "profile": profile if profile is not None else DEFAULT_PROFILE}

    # ------------------------------------------------------------------ persistence
    def save(self, path) -> Path:
        """Write momentum_policy.pt, momentum_config.json, feature_schema.json, preprocessing.pkl into `path`.

        momentum_policy.pt is self-contained and authoritative; the other three are standard
        side-car copies for tooling and the Meta-RL selector.
        """
        out = Path(path)
        out.mkdir(parents=True, exist_ok=True)
        schema = build_feature_schema(self.seq_len)
        pre_plain = {"feature_columns": self.pre["feature_columns"], "median": self.pre["median"].tolist(),
                     "mean": self.pre["mean"].tolist(), "std": self.pre["std"].tolist(), "clip": self.pre["clip"]}
        config = dict(self.config)
        config["temperature"] = self.temperature
        bundle = {
            "format_version": 1,
            "state_dict": {k: v.detach().cpu() for k, v in self.net.state_dict().items()},
            "config": config,
            "preprocessing": pre_plain,
            "feature_schema": schema,
        }
        torch.save(bundle, out / ARTIFACT_NAME)
        (out / "momentum_config.json").write_text(json.dumps(_jsonable(config), indent=2))
        (out / "feature_schema.json").write_text(json.dumps(_jsonable(schema), indent=2))
        with open(out / "preprocessing.pkl", "wb") as fh:
            pickle.dump({**pre_plain, "median": self.pre["median"], "mean": self.pre["mean"], "std": self.pre["std"]}, fh)
        return out / ARTIFACT_NAME

    @classmethod
    def load(cls, path, device=None) -> "MomentumPolicy":
        """Load from a directory containing momentum_policy.pt, or from the .pt file itself."""
        p = Path(path)
        pt = p / ARTIFACT_NAME if p.is_dir() else p
        if not pt.exists():
            raise FileNotFoundError(f"policy file not found: {pt}")
        device = torch.device(device) if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bundle = torch.load(pt, map_location=device, weights_only=True)
        config = bundle["config"]
        net = MomentumPolicyNet(**config["model"])
        net.load_state_dict(bundle["state_dict"])
        return cls(net, config, bundle["preprocessing"], device)


In [ ]:
import sp01_model
importlib.reload(sp01_model)
from sp01_model import MomentumPolicyNet, MomentumPolicy, mask_logits

set_seed(CONFIG["seed"])
MODEL_CFG = {"n_features": N_FEATURES, "n_static": N_STATIC, "n_actions": N_ACTIONS, **CONFIG["model"]}
net = MomentumPolicyNet(**MODEL_CFG).to(DEVICE)
print(net)
print(f"trainable parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")

## 7. Training
Supervised classification of the (proxy or genuine) action labels with a class-weighted cross-entropy on **position-masked logits**. Early stopping uses the validation loss; the best weights are restored. Windows are gathered on the fly from the scaled feature matrix (no giant window tensor is stored).

In [ ]:
X_ALL = torch.from_numpy(X_SCALED).to(DEVICE)
OFFSETS = torch.arange(-(L - 1), 1, device=DEVICE)
OFFSETS_NP = np.arange(-(L - 1), 1)

def to_torch(S):
    return {"row": torch.from_numpy(S["row"]).long().to(DEVICE), "static": torch.from_numpy(S["static"]).to(DEVICE),
            "y": torch.from_numpy(S["y"]).long().to(DEVICE), "valid": torch.from_numpy(S["valid"]).to(DEVICE)}

TS = {k: to_torch(v) for k, v in SAMPLES.items()}

def get_batch(T, idx):
    win = T["row"][idx].unsqueeze(1) + OFFSETS            # (B, L) global row ids, all inside one symbol block
    return X_ALL[win], T["static"][idx], T["valid"][idx], T["y"][idx]

@torch.no_grad()
def forward_all(model, T, batch_size=4096):
    """Masked logits for every sample of a split (CPU tensor)."""
    model.eval(); out = []
    for i in range(0, len(T["y"]), batch_size):
        idx = torch.arange(i, min(i + batch_size, len(T["y"])), device=DEVICE)
        seq, st, valid, _ = get_batch(T, idx)
        out.append(mask_logits(model(seq, st), valid).cpu())
    return torch.cat(out)

tc = CONFIG["train"]
class_w = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
optimizer = torch.optim.AdamW(net.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
y_val_cpu = TS["val"]["y"].cpu()
gen = torch.Generator().manual_seed(CONFIG["seed"])
n_train = len(TS["train"]["y"])

history, best_val, best_state, bad_epochs = [], float("inf"), None, 0
t0 = time.time()
for epoch in range(1, tc["epochs"] + 1):
    net.train()
    perm = torch.randperm(n_train, generator=gen).to(DEVICE)       # shuffling happens inside the TRAIN split only
    total, seen = 0.0, 0
    for i in range(0, n_train, tc["batch_size"]):
        idx = perm[i:i + tc["batch_size"]]
        seq, st, valid, y = get_batch(TS["train"], idx)
        if tc["input_noise_std"] > 0:
            seq = seq + tc["input_noise_std"] * torch.randn_like(seq)
        loss = Fnn.cross_entropy(mask_logits(net(seq, st), valid), y, weight=class_w)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), tc["grad_clip"])
        optimizer.step()
        total += loss.item() * len(idx); seen += len(idx)
    train_loss = total / seen
    val_logits = forward_all(net, TS["val"])
    val_loss = Fnn.cross_entropy(val_logits, y_val_cpu, weight=class_w.cpu()).item()
    val_f1 = f1_score(y_val_cpu.numpy(), val_logits.argmax(1).numpy(), average="macro", labels=list(range(N_ACTIONS)), zero_division=0)
    scheduler.step(val_loss)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_macro_f1": val_f1,
                    "lr": optimizer.param_groups[0]["lr"]})
    improved = val_loss < best_val - 1e-5
    if improved:
        best_val, bad_epochs = val_loss, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | val macro-F1 {val_f1:.3f} | lr {optimizer.param_groups[0]['lr']:.1e}"
          f"{'  *' if improved else ''}  [{time.time() - t0:.0f}s]")
    if bad_epochs >= tc["patience"]:
        print(f"early stopping at epoch {epoch} (no validation improvement for {tc['patience']} epochs)")
        break

net.load_state_dict(best_state); net.eval()
history_df = pd.DataFrame(history)
BEST_EPOCH = int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"])
print(f"\nbest epoch: {BEST_EPOCH} | best validation loss: {best_val:.4f}")

## 8. Calibration and classification metrics
* **Temperature scaling** is fit on the validation logits only (one scalar, minimising NLL) and then frozen.
* Metrics are reported on the chronological **validation** and **test** splits: accuracy, balanced accuracy, macro/weighted F1, per-class precision/recall/F1, confusion matrix, negative log-likelihood and Expected Calibration Error (ECE).
* Because the labels are proxy labels produced by a deterministic rule, these numbers measure **imitation fidelity**, not trading skill. The majority-class baseline is shown for reference.

In [ ]:
def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = Fnn.cross_entropy(logits / torch.exp(log_t), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05, 10.0).item())

def ece_score(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
            rows.append({"bin_lo": lo, "bin_hi": hi, "n": int(m.sum()), "confidence": float(conf[m].mean()), "accuracy": float(correct[m].mean())})
    return float(ece), rows

def classification_summary(logits, y, temperature):
    y = np.asarray(y); probs = torch.softmax(logits / temperature, dim=1).numpy()
    pred = probs.argmax(1); conf = probs.max(1); labels = list(range(N_ACTIONS))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    ece, bins = ece_score(conf, (pred == y).astype(float))
    return {"n_samples": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=labels, zero_division=0)),
            "weighted_f1": float(f1_score(y, pred, average="weighted", labels=labels, zero_division=0)),
            "majority_class_baseline_accuracy": float(np.bincount(y, minlength=N_ACTIONS).max() / len(y)),
            "nll": float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1.0)).mean()),
            "ece": ece, "reliability_bins": bins, "temperature": float(temperature),
            "per_class": {ACTION_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i in labels},
            "confusion_matrix": confusion_matrix(y, pred, labels=labels).tolist(), "pred": pred}

LOGITS = {k: forward_all(net, TS[k]) for k in ("val", "test")}
Y = {k: TS[k]["y"].cpu().numpy() for k in ("val", "test")}
TEMPERATURE = fit_temperature(LOGITS["val"], TS["val"]["y"].cpu())
print(f"fitted temperature (validation only): {TEMPERATURE:.3f}")

CLS = {k: classification_summary(LOGITS[k], Y[k], TEMPERATURE) for k in ("val", "test")}
CLS_UNCAL_TEST = classification_summary(LOGITS["test"], Y["test"], 1.0)
for k in ("val", "test"):
    c = CLS[k]
    print(f"\n[{k}] acc {c['accuracy']:.3f} | balanced acc {c['balanced_accuracy']:.3f} | macro-F1 {c['macro_f1']:.3f} | "
          f"weighted-F1 {c['weighted_f1']:.3f} | majority baseline {c['majority_class_baseline_accuracy']:.3f}")
    print(f"      NLL {c['nll']:.4f} | ECE {c['ece']:.4f}" + (f"  (uncalibrated test: NLL {CLS_UNCAL_TEST['nll']:.4f}, ECE {CLS_UNCAL_TEST['ece']:.4f})" if k == "test" else ""))
    print(pd.DataFrame(c["per_class"]).T.round(3).to_string())
    print("confusion matrix (rows = true, cols = predicted):")
    print(pd.DataFrame(c["confusion_matrix"], index=ACTION_NAMES, columns=ACTION_NAMES).to_string())

policy_config = {
    "model_name": "SP-01 Momentum Policy", "policy_id": "SP-01", "version": "1.0.0", "artifact": "momentum_policy.pt",
    "seq_len": L, "allow_short": CONFIG["allow_short"], "temperature": TEMPERATURE,
    "model": MODEL_CFG, "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
    "label_source": LABEL_SOURCE,
    "label_note": ("Labels are strategy-derived proxy labels from a rule-based momentum strategy, NOT human demonstrations."
                   if LABEL_SOURCE == "strategy_derived_proxy" else "Labels are genuine trader demonstrations."),
    "confidence_definition": "temperature-calibrated probability of the chosen action after masking invalid actions; calibrated against the training labels, not against trade profitability",
    "execution_convention": "decision at close of bar t, fill at open of bar t+1",
    "data": {"symbols": list(raw_data), "benchmark": CONFIG["benchmark"], "start_date": CONFIG["start_date"], "end_date": CONFIG["end_date"],
             "train_end": CONFIG["train_end"], "val_end": CONFIG["val_end"], "source": "yfinance daily adjusted OHLCV"},
    "training": {**CONFIG["train"], "seed": CONFIG["seed"], "best_epoch": BEST_EPOCH, "class_weights": CLASS_WEIGHTS.tolist(),
                 "n_train_samples": int(n_train)},
    "intended_use": "research and paper trading only; not a recommendation to trade real money",
}
policy = MomentumPolicy(net, policy_config, PREPROCESSING, DEVICE)
print("\nMomentumPolicy ready (in-memory).")

## 9. Chronological backtest (with position accounting)
Next-bar **action prediction** is kept separate from the **return actually earned** after the action can be executed:

* The policy sees bar *t* (close known) and chooses an action. It **fills at the open of bar *t+1*** (never at the close it observed).
* Return on day *t+1* for one symbol: `pos_old · (open_{t+1}/close_t − 1) + pos_new · (close_{t+1}/open_{t+1} − 1) − |pos_new − pos_old| · (fee + slippage) − short borrow`.
  The previous position earns the overnight gap, the new position earns the intraday move, and trades pay fees and slippage on traded notional.
* Positions are +1 / 0 / −1 of a fixed equal sleeve per symbol (portfolio return = mean over symbols); gross exposure never exceeds 100%. There is no leverage and no look-ahead.
* The policy is driven through the public `MomentumPolicy.predict` API with its **own** position path, so the backtest exercises exactly the inference code that gets shipped. Invalid actions are replaced by HOLD.
* Benchmarks: SPY buy-and-hold, equal-weight buy-and-hold of the universe (daily rebalanced, no costs) and the **rule-based teacher** applied directly (to show whether the network adds anything beyond the rule).
* The test split is untouched by training, calibration and early stopping. It is still a single market regime and a hand-picked list of survivors, so treat the numbers as a plumbing check and weak evidence, not proof of edge.

In [ ]:
OPEN_ARR = panel["Open"].to_numpy(dtype=float); CLOSE_ARR = panel["Close"].to_numpy(dtype=float)
PROFILES = {"balanced": (0.5, 0.5, 0.5), "conservative": (0.2, 0.8, 0.2), "aggressive": (0.8, 0.3, 0.8)}
COST_RATE = (CONFIG["fee_bps"] + CONFIG["slippage_bps"]) / 1e4
BORROW_DAILY = CONFIG["short_borrow_bps_annual"] / 1e4 / 252.0

def prepare_backtest(split: str) -> dict:
    rows = eligible_rows(split)
    sub = panel.loc[rows, ["symbol", "date"]].assign(row=rows)
    piv = sub.pivot(index="date", columns="symbol", values="row")
    syms = list(piv.columns)
    n_all = len(piv); piv = piv.dropna(how="any")
    if len(piv) < n_all:
        print(f"  [{split}] {n_all - len(piv)} dates dropped because not every symbol had a complete window")
    rows_mat = piv[syms].to_numpy().astype(np.int64)
    bclose = bench_df["Close"].reindex(piv.index).ffill(limit=3).to_numpy(dtype=float)
    if np.isnan(bclose).any():
        raise RuntimeError("benchmark close has gaps inside the backtest window")
    return {"split": split, "dates": piv.index, "symbols": syms, "rows": rows_mat,
            "open": OPEN_ARR[rows_mat], "close": CLOSE_ARR[rows_mat], "bench_close": bclose}

def run_backtest(bt: dict, policy_fn) -> dict:
    T, S = bt["open"].shape
    pos = np.zeros(S, dtype=int)
    rets = np.zeros((T - 1, S)); posn = np.zeros((T - 1, S), dtype=int); turn = np.zeros((T - 1, S)); n_trades = 0
    for k in range(T - 1):
        action = np.asarray(policy_fn(k, bt["rows"][k], pos)).astype(int)
        valid = valid_action_mask(pos, CONFIG["allow_short"])
        action = np.where(valid[np.arange(S), action], action, HOLD)
        new = apply_action(pos, action)
        gap = bt["open"][k + 1] / bt["close"][k] - 1.0                 # earned by the OLD position
        intra = bt["close"][k + 1] / bt["open"][k + 1] - 1.0           # earned by the NEW position (filled at the open)
        traded = np.abs(new - pos)
        rets[k] = pos * gap + new * intra - traded * COST_RATE - (new < 0) * BORROW_DAILY
        posn[k], turn[k] = new, traded
        n_trades += int((traded > 0).sum()); pos = new
    port = rets.mean(axis=1)
    return {"dates": bt["dates"][1:], "ret": port, "equity": np.cumprod(1.0 + port), "positions": posn,
            "turnover": turn.mean(axis=1), "exposure": np.abs(posn).mean(axis=1), "n_trades": n_trades, "per_symbol_ret": rets}

def perf_metrics(r, turnover=None, exposure=None, n_trades=None) -> dict:
    r = np.asarray(r, dtype=float); n = len(r); eq = np.cumprod(1.0 + r)
    years = n / 252.0; sd = r.std(ddof=1) if n > 1 else np.nan
    ex = r - CONFIG["risk_free_annual"] / 252.0
    down = np.minimum(ex, 0.0); dd_sd = np.sqrt((down ** 2).mean()) if n else np.nan
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    mdd = float((eq / peak - 1.0).min())
    cagr = float(eq[-1] ** (1 / years) - 1) if years > 0 and eq[-1] > 0 else float("nan")
    out = {"n_days": int(n), "total_return": float(eq[-1] - 1.0), "cagr": cagr,
           "ann_vol": float(sd * np.sqrt(252)) if sd == sd else float("nan"),
           "sharpe": float(ex.mean() / sd * np.sqrt(252)) if sd and sd > 0 else float("nan"),
           "sortino": float(ex.mean() / dd_sd * np.sqrt(252)) if dd_sd and dd_sd > 0 else float("nan"),
           "max_drawdown": mdd, "calmar": float(cagr / abs(mdd)) if mdd < 0 and cagr == cagr else float("nan")}
    out["avg_daily_turnover"] = float(np.mean(turnover)) if turnover is not None else 0.0
    out["annualized_turnover"] = out["avg_daily_turnover"] * 252.0
    out["avg_gross_exposure"] = float(np.mean(exposure)) if exposure is not None else 1.0
    out["n_trades"] = int(n_trades) if n_trades is not None else 0
    return out

def model_policy_fn(profile):
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos):
        seq = RAW_MAT[rows[:, None] + OFFSETS_NP]                       # (S, L, F) raw, past-only windows
        return policy.predict({"sequence": seq, "position": pos, "profile": prof})["action"]
    return fn

def teacher_policy_fn(profile):
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos):
        return momentum_rule_labels({c: v[rows] for c, v in RAW.items()}, pos, np.tile(prof, (len(rows), 1)), CONFIG["allow_short"])
    return fn

BT, BT_RESULTS, BT_TABLE = {}, {}, {}
for split in ("val", "test"):
    bt = prepare_backtest(split); BT[split] = bt
    res = {}
    for pname, prof in PROFILES.items():
        res[f"SP-01 model ({pname})"] = run_backtest(bt, model_policy_fn(prof))
    res["Rule teacher (balanced)"] = run_backtest(bt, teacher_policy_fn(PROFILES["balanced"]))
    dates = res["SP-01 model (balanced)"]["dates"]
    b_ret = bt["bench_close"][1:] / bt["bench_close"][:-1] - 1.0
    ew_ret = (bt["close"][1:] / bt["close"][:-1] - 1.0).mean(axis=1)
    metrics = {name: perf_metrics(r["ret"], r["turnover"], r["exposure"], r["n_trades"]) for name, r in res.items()}
    metrics[f"{CONFIG['benchmark']} buy & hold"] = perf_metrics(b_ret)
    metrics["Equal-weight universe buy & hold"] = perf_metrics(ew_ret)
    BT_RESULTS[split] = {"runs": res, "bench_ret": b_ret, "ew_ret": ew_ret, "dates": dates}
    BT_TABLE[split] = pd.DataFrame(metrics).T
    print(f"\n===== {split.upper()} backtest: {dates[0].date()} -> {dates[-1].date()} ({len(dates)} trading days, {len(bt['symbols'])} symbols) =====")
    print(f"costs: fee {CONFIG['fee_bps']} bps + slippage {CONFIG['slippage_bps']} bps per side, short borrow {CONFIG['short_borrow_bps_annual']} bps/yr")
    print(BT_TABLE[split][["total_return", "cagr", "ann_vol", "sharpe", "max_drawdown", "annualized_turnover", "avg_gross_exposure", "n_trades"]].round(3).to_string())

In [ ]:
# ---- Next-bar action prediction vs the return that can actually be earned after the action ----------------
# For every TEST bar, flat-state, balanced profile: what did the executable 5-day return (open t+1 -> open t+6) look like
# after each predicted action? (fwd_open_ret_5 is an evaluation-only column and never a model input.)
test_rows = eligible_rows("test")
preds = []
for i in range(0, len(test_rows), 2048):
    r = test_rows[i:i + 2048]
    out = policy.predict({"sequence": RAW_MAT[r[:, None] + OFFSETS_NP], "position": 0, "profile": PROFILES["balanced"]})
    preds.append(out["action"])
preds = np.concatenate(preds)
fwd = panel.loc[test_rows, "fwd_open_ret_5"].to_numpy()
ok = np.isfinite(fwd)
tab = pd.DataFrame({"action": [ACTION_NAMES[a] for a in preds[ok]], "fwd": fwd[ok]})
exec_table = tab.groupby("action")["fwd"].agg(n="count", mean_fwd_5d_return="mean", hit_rate_positive=lambda s: (s > 0).mean())
exec_table.loc["ALL BARS (unconditional)"] = [len(tab), tab["fwd"].mean(), (tab["fwd"] > 0).mean()]
if "SELL" in exec_table.index:
    exec_table["direction_adjusted_mean"] = [(-1 if i == "SELL" else 1) * v for i, v in zip(exec_table.index, exec_table["mean_fwd_5d_return"])]
print("Executable 5-day return (fill at next open) by predicted action, flat state, TEST split:")
print(exec_table.round(4).to_string())
print("\nA predicted BUY only has predictive value if its mean forward return beats the unconditional row; "
      "statistical significance is NOT established by this table.")

## 10. Plots

In [ ]:
FIG_DIR = ART_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train loss")
ax[0].plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
ax[0].axvline(BEST_EPOCH, color="grey", ls="--", lw=1, label=f"best epoch ({BEST_EPOCH})")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("weighted cross-entropy"); ax[0].set_title("Training and validation loss"); ax[0].legend()
ax[1].plot(history_df["epoch"], history_df["val_macro_f1"], color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("macro F1"); ax[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.savefig(FIG_DIR / "loss_curves.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
cm = np.array(CLS["test"]["confusion_matrix"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
for a, mat, title, fmt in ((ax[0], cm, "Test confusion matrix (counts)", "d"), (ax[1], cmn, "Test confusion matrix (row-normalised)", ".2f")):
    im = a.imshow(mat, cmap="Blues"); a.set_title(title)
    a.set_xticks(range(N_ACTIONS)); a.set_yticks(range(N_ACTIONS)); a.set_xticklabels(ACTION_NAMES); a.set_yticklabels(ACTION_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true (proxy label)")
    for i in range(N_ACTIONS):
        for j in range(N_ACTIONS):
            a.text(j, i, format(mat[i, j], fmt), ha="center", va="center", color="white" if mat[i, j] > mat.max() / 2 else "black")
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix_test.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(4.8, 4.5))
bins = pd.DataFrame(CLS["test"]["reliability_bins"])
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
ax.plot(bins["confidence"], bins["accuracy"], "o-", label=f"test (ECE {CLS['test']['ece']:.3f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy vs proxy labels"); ax.set_title("Reliability diagram"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "reliability_test.png", dpi=130); plt.show()

R = BT_RESULTS["test"]; main = R["runs"]["SP-01 model (balanced)"]; teacher = R["runs"]["Rule teacher (balanced)"]
d = R["dates"]
fig, ax = plt.subplots(3, 1, figsize=(11, 11), sharex=True, gridspec_kw={"height_ratios": [3, 1.6, 1.4]})
ax[0].plot(d, main["equity"], label="SP-01 model (balanced)", lw=2)
ax[0].plot(d, teacher["equity"], label="Rule teacher", lw=1.2, alpha=0.8)
ax[0].plot(d, np.cumprod(1 + R["bench_ret"]), label=f"{CONFIG['benchmark']} buy & hold", lw=1.2)
ax[0].plot(d, np.cumprod(1 + R["ew_ret"]), label="Equal-weight universe buy & hold", lw=1.2)
ax[0].set_ylabel("growth of 1.0"); ax[0].set_title("TEST split (out-of-sample): equity curves after costs"); ax[0].legend()
for name, eq in (("SP-01 model", main["equity"]), (CONFIG["benchmark"], np.cumprod(1 + R["bench_ret"]))):
    ax[1].plot(d, eq / np.maximum.accumulate(eq) - 1, label=name)
ax[1].set_ylabel("drawdown"); ax[1].legend()
ax[2].plot(d, main["exposure"], color="tab:purple"); ax[2].set_ylabel("gross exposure"); ax[2].set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig(FIG_DIR / "backtest_test.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
dist_pct = dist / dist.sum() * 100
dist_pct.plot(kind="bar", ax=ax); ax.set_ylabel("% of samples"); ax.set_title(f"Label distribution ({LABEL_SOURCE})"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(FIG_DIR / "label_distribution.png", dpi=130); plt.show()

## 11. Save standardized artifacts
Written to the artifact directory:

| File | Content |
|---|---|
| `momentum_policy.pt` | weights + config + preprocessing + feature schema (self-contained, loads with `weights_only=True`) |
| `momentum_config.json` | model/inference configuration (side-car copy) |
| `feature_schema.json` | exact input ordering and dimensions (side-car copy) |
| `preprocessing.pkl` | train-fit median / mean / std / clip (plain dict, no custom classes) |
| `evaluation_metrics.json` | classification, calibration, backtest and benchmark metrics |
| `equity_curve.csv` | chronological backtest equity curves (validation and test, each starting at 1.0) |
| `sp01_features.py`, `sp01_model.py` | the exact feature and model code needed to use the policy |

`momentum_policy.pt` is authoritative; the other copies exist for inspection and tooling.

In [ ]:
def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Timestamp,)): return str(o.date())
    return o

# 1-4: policy.pt, config.json, feature_schema.json, preprocessing.pkl
pt_path = policy.save(ART_DIR)

# 5: evaluation_metrics.json
def strip_pred(c): return {k: v for k, v in c.items() if k != "pred"}
evaluation = {
    "model": "SP-01 Momentum Policy", "label_source": LABEL_SOURCE,
    "label_warning": policy_config["label_note"],
    "disclaimer": "Research / paper-trading model. Backtest numbers are a single historical path, are not evidence of future profitability, and are not a recommendation to trade real money.",
    "data": policy_config["data"], "n_samples": {k: int(len(v["y"])) for k, v in SAMPLES.items()},
    "label_distribution": dist.to_dict(), "class_weights": dict(zip(ACTION_NAMES, CLASS_WEIGHTS.tolist())),
    "classification": {"validation": strip_pred(CLS["val"]), "test": strip_pred(CLS["test"]),
                       "test_uncalibrated": strip_pred(CLS_UNCAL_TEST)},
    "backtest_assumptions": {"execution": "decision at close t, fill at open t+1", "fee_bps": CONFIG["fee_bps"],
                             "slippage_bps": CONFIG["slippage_bps"], "short_borrow_bps_annual": CONFIG["short_borrow_bps_annual"],
                             "risk_free_annual": CONFIG["risk_free_annual"], "allow_short": CONFIG["allow_short"],
                             "sizing": "equal fixed sleeve per symbol, +/-1 position, no leverage"},
    "backtest": {s: BT_TABLE[s].to_dict(orient="index") for s in ("val", "test")},
    "next_bar_vs_executable_return_test": exec_table.to_dict(orient="index"),
    "training_history": history_df.to_dict(orient="records"), "best_epoch": BEST_EPOCH,
}
(ART_DIR / "evaluation_metrics.json").write_text(json.dumps(to_py(evaluation), indent=2))

# 6: equity_curve.csv
curves = []
for split in ("val", "test"):
    R_ = BT_RESULTS[split]; m_ = R_["runs"]["SP-01 model (balanced)"]; t_ = R_["runs"]["Rule teacher (balanced)"]
    curves.append(pd.DataFrame({
        "date": R_["dates"], "split": split, "strategy_return": m_["ret"], "strategy_equity": m_["equity"],
        "benchmark_return": R_["bench_ret"], "benchmark_equity": np.cumprod(1 + R_["bench_ret"]),
        "equal_weight_bh_equity": np.cumprod(1 + R_["ew_ret"]), "rule_teacher_equity": t_["equity"],
        "gross_exposure": m_["exposure"], "turnover": m_["turnover"]}))
pd.concat(curves, ignore_index=True).to_csv(ART_DIR / "equity_curve.csv", index=False)

# ship the exact code with the weights
for fname in ("sp01_features.py", "sp01_model.py"):
    shutil.copy(fname, ART_DIR / fname)

# round-trip check: reload from disk and compare with the in-memory policy
reloaded = MomentumPolicy.load(ART_DIR, DEVICE)
_r = test_rows[:256]
_state = {"sequence": RAW_MAT[_r[:, None] + OFFSETS_NP], "position": 0, "profile": PROFILES["balanced"]}
_a, _b = policy.predict(_state), reloaded.predict(_state)
assert np.allclose(_a["action_probs"], _b["action_probs"], atol=1e-5) and (_a["action"] == _b["action"]).all(), "reload mismatch!"
print("Round-trip check passed: reloaded policy reproduces in-memory predictions.")

ARTIFACTS = ["momentum_policy.pt", "momentum_config.json", "feature_schema.json", "preprocessing.pkl",
             "evaluation_metrics.json", "equity_curve.csv", "sp01_features.py", "sp01_model.py"]
for a in ARTIFACTS:
    assert (ART_DIR / a).exists(), f"missing artifact {a}"
    print(f"saved {(ART_DIR / a).resolve()}  ({(ART_DIR / a).stat().st_size / 1024:.1f} KiB)")

shutil.make_archive(str(ART_DIR), "zip", ART_DIR)
print(f"zip bundle: {Path(str(ART_DIR) + '.zip').resolve()}")
try:
    from google.colab import files  # noqa: F401  (only exists in Colab)
    files.download(str(ART_DIR) + ".zip")
except Exception:  # noqa: BLE001
    print("(not running in Colab or download blocked: copy the zip from the path above)")

## 12. Quick-start inference
This cell **loads the saved policy from disk** and predicts. It does **not** download data and does **not** train. The only inputs are a feature window, the current position and a trader profile. The Meta-RL environment can build the window itself, or use `policy.state_from_ohlcv(...)` on OHLCV frames it already holds.

In [ ]:
policy_loaded = MomentumPolicy.load(ART_DIR, DEVICE)       # or MomentumPolicy.load("path/to/momentum_policy.pt", "cpu")
assert "yfinance" not in Path("sp01_model.py").read_text() and "yfinance" not in Path("sp01_features.py").read_text(), \
    "inference modules must not depend on a data downloader"

# (a) single state from raw OHLCV frames already in memory (last bar of the first symbol)
sym0 = list(raw_data)[0]
state = policy_loaded.state_from_ohlcv(raw_data[sym0], bench_df, position=0, profile={"risk_tolerance": 0.5, "patience": 0.5, "aggressiveness": 0.5})
out = policy_loaded.predict(state)
print(f"{sym0} @ {raw_data[sym0].index[-1].date()} | flat | action = {out['action_name']} (id {out['action']}) | confidence = {out['confidence']:.3f}")
print("action probabilities:", {n: round(float(p), 3) for n, p in zip(ACTION_NAMES, out["action_probs"])}, "| valid:", {n: bool(v) for n, v in zip(ACTION_NAMES, out["valid_actions"])})

# (b) the same bar if we already hold a long position (different valid-action set)
out_long = policy_loaded.predict({**state, "position": 1})
print(f"same bar, currently LONG -> action = {out_long['action_name']} | confidence = {out_long['confidence']:.3f} | valid: { {n: bool(v) for n, v in zip(ACTION_NAMES, out_long['valid_actions'])} }")

# (c) batch of 5 test windows with an aggressive profile
r = test_rows[:5]
batch = policy_loaded.predict({"sequence": RAW_MAT[r[:, None] + OFFSETS_NP], "position": [0, 0, 1, 1, -1], "profile": PROFILES["aggressive"]})
print("batch actions:", batch["action_name"], "| confidences:", [round(float(c), 3) for c in batch["confidence"]])

## 13. Compatibility summary for the future Meta-RL selector

In [ ]:
test_m = BT_TABLE["test"]
print("=" * 92)
print("ARTIFACT PATHS")
for a in ARTIFACTS:
    print(f"  {(ART_DIR / a).resolve()}")
print(f"  {Path(str(ART_DIR) + '.zip').resolve()}   (zip of everything above)")
print("=" * 92)
print("COMPATIBILITY SUMMARY: SP-01 Momentum Policy")
print(f"  policy id / version     : SP-01 / {policy_config['version']}")
print(f"  action space (shared)   : {dict(enumerate(ACTION_NAMES))}")
print(f"  sequence input          : (batch, {L}, {N_FEATURES}) RAW features in sp01_features.FEATURE_COLUMNS order, oldest bar first")
print(f"  static input            : position in {{-1,0,+1}} (encoded flat/long/short) + profile {PROFILE_FEATURES} in [0,1]  -> {N_STATIC} dims")
print( "  call                    : policy.predict({'sequence': ..., 'position': ..., 'profile': ...}) -> action_probs(4), action, action_name, confidence, valid_actions, entropy")
print( "  valid actions           : flat -> HOLD/BUY/SELL(if allow_short); long or short -> HOLD/EXIT (invalid actions get probability 0)")
print(f"  allow_short             : {CONFIG['allow_short']}")
print( "  confidence              : temperature-calibrated max probability over VALID actions (agreement with the training labels, not P(profit))")
print(f"  execution convention   : decision at close of bar t, fill at open of bar t+1 (matches the backtest)")
print(f"  labels                  : {LABEL_SOURCE}" + ("  -> proxy labels from a rule-based strategy, NOT human demonstrations" if LABEL_SOURCE == 'strategy_derived_proxy' else ""))
print( "  inference deps          : torch, numpy, pandas only (no yfinance, no training code)")
print( "  loading                 : MomentumPolicy.load(dir_or_pt, device) with sp01_features.py and sp01_model.py on PYTHONPATH")
print("-" * 92)
print("OUT-OF-SAMPLE (TEST) SNAPSHOT: one regime, one symbol list, costs included; not proof of profitability")
snap = test_m.loc[["SP-01 model (balanced)", "Rule teacher (balanced)", f"{CONFIG['benchmark']} buy & hold", "Equal-weight universe buy & hold"],
                  ["total_return", "sharpe", "max_drawdown", "annualized_turnover"]].round(3)
print(snap.to_string())
print("-" * 92)
print("For research and paper trading only. This is not a recommendation to trade real money.")
print("=" * 92)